# 04 · Evaluation

Evaluates the forecasts of `02_baselines` and `03_fusion` with **conformal prediction** and then tests the selected models on the test set. It has two stages:

| Stage | Data | What is done |
|---|---|---|
| 1 | out-of-fold predictions of `02_baselines` and `03_fusion` (cross-validation) | every model is calibrated to 90% coverage, then the models are compared and the two best single-modality and the two best fusion models are selected |
| 2 | test set (used **once**) | the selected models are retrained on the whole training set and predict the test set; their intervals are calibrated with the out-of-fold scores |

Stage 1 trains nothing and does not use the test set.

In [50]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats, special

# paths
PROCESSED_DIR = Path("data") / "processed"
PRED_DIRS = [PROCESSED_DIR / "baselines" / "predictions", PROCESSED_DIR / "fusion" / "predictions"]
RESULTS_DIR = PROCESSED_DIR / "evaluation"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# problem setup (identical to 02_baselines and 03_fusion)
HORIZONS = [15, 30, 60]
N_FOLDS = 5
WEATHER_BINS = ["clear", "variable", "overcast"]
ALPHA = 0.1                      # 90% prediction intervals

## 1 · Models

All models of `02_baselines` and `03_fusion`, in four groups: the two persistence references, the single-modality models and the fusion models (the **candidates** for the test set), and the routing and shuffled-image tests of `03_fusion` (analysis only, not candidates).

In [51]:
# model name -> (group, label); "single" and "fusion" are the candidates for the test set
MODELS = {
    "persistence": ("reference", "Persistence"),
    "smart_persistence": ("reference", "Smart persistence"),
    "pv_mlp": ("single", "PV history – MLP"),
    "pv_lstm": ("single", "PV history – LSTM"),
    "pv_transformer": ("single", "PV history – Transformer"),
    "k_mlp": ("single", "Clear-sky index history – MLP"),
    "k_lstm": ("single", "Clear-sky index history – LSTM"),
    "k_transformer": ("single", "Clear-sky index history – Transformer"),
    "sky_cnn": ("single", "Sky images – CNN"),
    "fusion_concat": ("fusion", "Concatenation"),
    "fusion_residual": ("fusion", "Residual (two-step)"),
    "fusion_film": ("fusion", "FiLM (early fusion)"),
    "fusion_xattn": ("fusion", "Cross-attention"),
    "fusion_pretrained": ("fusion", "Concatenation, pre-trained encoder"),
    "fusion_xattn_images_to_mu": ("analysis", "Cross-attention, images to μ only"),
    "fusion_xattn_images_to_sigma": ("analysis", "Cross-attention, images to σ only"),
    "fusion_xattn_images_shuffled_in_training": ("analysis", "Cross-attention, trained with shuffled images"),
    "fusion_xattn_images_shuffled_at_test": ("analysis", "Cross-attention, images shuffled at test"),
}
LABELS = {m: label for m, (group, label) in MODELS.items()}

def load_predictions(model, h):
    # out-of-fold predictions saved by 02_baselines or 03_fusion (None if this model was not run at h)
    for d in PRED_DIRS:
        f = d / f"{model}_h{h}.csv"
        if f.exists():
            return pd.read_csv(f)
    return None

## 2 · Conformal calibration

The 90% interval of a model is μ ± t₀.₉₅(ν) · σ, but its real coverage is not exactly 90%. Conformal prediction replaces t₀.₉₅(ν) by a factor q̂ learned from errors on unseen data:

1. on calibration samples, compute the score s = |y − μ| / σ (how many σ the true value is away from μ);
2. q̂ is the 90% quantile of the scores (split-conformal rank ⌈(n + 1) · 0.9⌉);
3. the interval of a new sample is μ ± q̂ · σ. μ does not change, so RMSE, CRPS and NLL do not change either.

Two versions: **global** (one q̂) and **weather-conditional** (one q̂ per weather bin, so that every weather bin is covered at 90%; the weather bin is computed from the last 15 minutes of k, so it is known at forecast time). The weather-conditional version is the main method.

In stage 1, the q̂ for fold f is computed from the out-of-fold scores of the other 4 folds, so the calibration never sees the evaluated fold. This is the same situation as in stage 2, where q̂ from the fold models is used for a new model trained on the whole training set.

In [52]:
# find the 90% quantile
def conformal_quantile(scores):
    # split-conformal quantile: the ceil((n + 1)(1 - alpha))-th smallest score
    n = len(scores)
    k = min(int(np.ceil((n + 1) * (1 - ALPHA))), n)
    return np.sort(scores)[k - 1]

def intervals(pred, calibration):
    # 90% interval of every sample: "raw" = Student-t interval of the model,
    # "global" / "weather" = conformal, q for fold f from the scores of the other 4 folds
    if calibration == "raw":
        q = stats.t.ppf(1 - ALPHA / 2, pred["nu"].to_numpy())
    else:
        score = ((pred["y"] - pred["mu"]).abs() / pred["sigma"]).to_numpy()
        q = np.full(len(pred), np.nan)
        groups = WEATHER_BINS if calibration == "weather" else [None]
        for fold in range(N_FOLDS):
            for w in groups:
                same = np.ones(len(pred), bool) if w is None else (pred["weather"] == w).to_numpy()
                calib = same & (pred["fold"] != fold).to_numpy()
                target = same & (pred["fold"] == fold).to_numpy()
                q[target] = conformal_quantile(score[calib])
    return pred["mu"].to_numpy() - q * pred["sigma"].to_numpy(), pred["mu"].to_numpy() + q * pred["sigma"].to_numpy()

## 3 · Metrics

The same metrics as in `02_baselines`, in kW: RMSE, MAE, skill (1 − RMSE / RMSE of smart persistence), CRPS and NLL of the Student-t forecast, and coverage, width and Winkler score of the 90% interval, for each of the three intervals (raw, global, weather-conditional).

In [53]:
# CRPS calculation
def crps_student_t(z, nu):
    return (z * (2 * stats.t.cdf(z, nu) - 1)
            + 2 * stats.t.pdf(z, nu) * (nu + z ** 2) / (nu - 1)
            - 2 * np.sqrt(nu) * special.beta(0.5, nu - 0.5) / ((nu - 1) * special.beta(0.5, nu / 2) ** 2))

# all other metrics
def evaluate(pred):
    # metrics per fold, weather bin and calibration; RMSE, MAE, CRPS and NLL do not depend on the calibration
    y, mu, sigma, nu = (pred[c].to_numpy() for c in ["y", "mu", "sigma", "nu"])
    z = (y - mu) / sigma
    crps, nll = sigma * crps_student_t(z, nu), np.log(sigma) - stats.t.logpdf(z, nu)
    bounds = {c: intervals(pred, c) for c in ["raw", "global", "weather"]}
    rows = []
    for fold in range(N_FOLDS):
        for weather in ["all"] + WEATHER_BINS:
            m = (pred["fold"] == fold).to_numpy() & ((pred["weather"] == weather).to_numpy() | (weather == "all"))
            point = {"RMSE": np.sqrt(np.mean((y[m] - mu[m]) ** 2)), "MAE": np.mean(np.abs(y[m] - mu[m])),
                     "CRPS": crps[m].mean(), "NLL": nll[m].mean()}
            for calibration, (low, high) in bounds.items():
                lo, hi, yy = low[m], high[m], y[m]
                miss = (lo - yy) * (yy < lo) + (yy - hi) * (yy > hi)
                rows.append({"fold": fold, "weather": weather, "calibration": calibration, **point,
                             "coverage": np.mean((yy >= lo) & (yy <= hi)), "width": np.mean(hi - lo),
                             "Winkler": np.mean(hi - lo + 2 / ALPHA * miss)})
    return pd.DataFrame(rows)

## 4 · Stage 1: conformal calibration of the cross-validation predictions

Every model at every horizon, per fold and weather bin. The results are saved to `evaluation/cv_conformal_metrics.csv`.

In [54]:
results = []
for h in HORIZONS:
    for model in MODELS:
        pred = load_predictions(model, h)
        if pred is not None:
            r = evaluate(pred)
            r.insert(0, "model", model)
            r.insert(1, "horizon", h)
            results.append(r)
results = pd.concat(results, ignore_index=True)

# skill score against smart persistence, same horizon, fold and weather bin
key = ["horizon", "fold", "weather", "calibration"]
reference = results[results["model"] == "smart_persistence"].set_index(key)["RMSE"]
results["skill"] = 1 - results["RMSE"] / reference.reindex(pd.MultiIndex.from_frame(results[key])).to_numpy()
results.to_csv(RESULTS_DIR / "cv_conformal_metrics.csv", index=False)
print(f"{results['model'].nunique()} models, {len(results)} rows")

18 models, 3120 rows


### 4.1 Single-modality models

All metrics after weather-conditional calibration (mean over the 5 folds, all weather), with the two persistence references at the top. Every interval now covers 90%, so the interval width and the Winkler score can be compared directly.

In [55]:
METRICS = ["RMSE", "MAE", "skill", "CRPS", "NLL", "coverage", "width", "Winkler"]
CANDIDATES = [m for m, (g, _) in MODELS.items() if g != "analysis"]

def summary(models):
    # all metrics, mean over the 5 folds, all weather, weather-conditional conformal intervals
    r = results[(results["calibration"] == "weather") & (results["weather"] == "all") & results["model"].isin(models)]
    t = r.groupby(["horizon", "model"], sort=False)[METRICS].mean().rename(index=LABELS, level="model").round(3)
    for h in HORIZONS:
        print(f"\n{h} minutes")
        display(t.loc[h])

def by_weather(metric, calibration):
    # one metric per horizon and weather bin, mean over the 5 folds
    r = results[(results["calibration"] == calibration) & results["model"].isin(CANDIDATES)]
    t = r.groupby(["model", "horizon", "weather"], sort=False)[metric].mean().unstack(["horizon", "weather"])
    return t.reindex(columns=["all"] + WEATHER_BINS, level=1).rename(index=LABELS).round(3)

summary([m for m, (g, _) in MODELS.items() if g in ("reference", "single")])


15 minutes


,RMSE,MAE,skill,CRPS,NLL,coverage,width,Winkler
model,,,,,,,,
Persistence,2.568,1.462,-0.079,1.084,1.883,0.900,6.346,9.778
Smart persistence,2.379,0.904,0.000,0.740,0.848,0.900,4.599,8.393
PV history – MLP,2.102,0.801,0.116,0.603,0.380,0.900,3.672,5.976
PV history – LSTM,2.085,0.798,0.123,0.600,0.390,0.900,3.712,5.941
PV history – Transformer,2.105,0.815,0.115,0.612,0.480,0.900,3.853,6.144
Clear-sky index history – MLP,2.082,0.791,0.125,0.593,0.366,0.900,3.593,5.833
Clear-sky index history – LSTM,2.074,0.785,0.128,0.589,0.360,0.900,3.569,5.833
Clear-sky index history – Transformer,2.081,0.797,0.125,0.597,0.384,0.900,3.600,5.910
Sky images – CNN,2.115,1.008,0.111,0.737,1.189,0.899,4.316,6.156



30 minutes


,RMSE,MAE,skill,CRPS,NLL,coverage,width,Winkler
model,,,,,,,,
Persistence,3.308,2.317,-0.230,1.650,2.419,0.900,9.359,12.894
Smart persistence,2.690,1.151,0.000,0.913,1.243,0.900,5.622,9.754
PV history – MLP,2.373,0.975,0.118,0.738,0.709,0.900,4.611,7.571
PV history – LSTM,2.356,0.973,0.124,0.736,0.720,0.900,4.606,7.524
PV history – Transformer,2.364,0.982,0.121,0.743,0.783,0.900,4.660,7.653
Clear-sky index history – MLP,2.330,0.958,0.134,0.720,0.689,0.900,4.356,7.197
Clear-sky index history – LSTM,2.346,0.959,0.128,0.723,0.686,0.900,4.517,7.340
Clear-sky index history – Transformer,2.357,0.966,0.124,0.728,0.696,0.900,4.505,7.410
Sky images – CNN,2.349,1.121,0.127,0.824,1.278,0.899,4.712,7.058



60 minutes


,RMSE,MAE,skill,CRPS,NLL,coverage,width,Winkler
model,,,,,,,,
Persistence,4.896,3.919,-0.558,2.703,2.948,0.900,14.851,18.177
Smart persistence,3.146,1.538,0.000,1.178,1.665,0.900,7.072,11.460
PV history – MLP,2.658,1.164,0.155,0.886,1.023,0.900,5.477,9.289
PV history – LSTM,2.685,1.182,0.147,0.902,1.057,0.900,5.819,9.624
PV history – Transformer,2.672,1.181,0.151,0.898,1.057,0.900,5.886,9.602
Clear-sky index history – MLP,2.648,1.159,0.158,0.878,1.015,0.899,5.397,9.067
Clear-sky index history – LSTM,2.643,1.157,0.160,0.880,1.020,0.900,5.603,9.224
Clear-sky index history – Transformer,2.665,1.164,0.153,0.884,1.021,0.899,5.500,9.242
Sky images – CNN,2.628,1.258,0.165,0.930,1.379,0.896,5.634,8.508


### 4.2 Fusion models

The same metrics for the fusion models of `03_fusion`.

In [56]:
summary([m for m, (g, _) in MODELS.items() if g == "fusion"])


15 minutes


,RMSE,MAE,skill,CRPS,NLL,coverage,width,Winkler
model,,,,,,,,
Concatenation,1.932,0.739,0.188,0.545,0.292,0.900,3.180,4.966
Residual (two-step),1.958,0.747,0.177,0.553,0.302,0.900,3.242,5.085
FiLM (early fusion),1.968,0.746,0.173,0.552,0.305,0.900,3.258,5.104
Cross-attention,1.946,0.739,0.182,0.546,0.284,0.900,3.213,5.032
"Concatenation, pre-trained encoder",2.013,0.769,0.154,0.571,0.347,0.899,3.351,5.328



30 minutes


,RMSE,MAE,skill,CRPS,NLL,coverage,width,Winkler
model,,,,,,,,
Concatenation,2.182,0.893,0.189,0.660,0.593,0.899,3.875,6.078
Residual (two-step),2.204,0.895,0.181,0.666,0.597,0.899,4.061,6.336
FiLM (early fusion),2.220,0.904,0.175,0.672,0.617,0.899,4.055,6.315
Cross-attention,2.191,0.890,0.186,0.660,0.583,0.900,3.913,6.150
"Concatenation, pre-trained encoder",2.265,0.924,0.158,0.686,0.644,0.899,3.933,6.383



60 minutes


,RMSE,MAE,skill,CRPS,NLL,coverage,width,Winkler
model,,,,,,,,
Concatenation,2.495,1.075,0.207,0.805,0.906,0.900,4.899,7.837
Residual (two-step),2.510,1.091,0.202,0.814,0.920,0.898,5.099,7.913
FiLM (early fusion),2.580,1.114,0.180,0.832,0.935,0.899,4.956,7.942
Cross-attention,2.492,1.084,0.208,0.806,0.918,0.900,4.759,7.641
"Concatenation, pre-trained encoder",2.584,1.117,0.179,0.838,0.972,0.899,4.939,8.082


### 4.3 Width by weather

Width of the weather-conditional interval (kW) in each weather bin: where are the intervals narrower?

In [57]:
display(by_weather("width", "weather"))

horizon                                   15                              30  \
weather                                  all  clear variable overcast    all   
model                                                                          
Persistence                            6.346  3.202   14.212   10.045  9.359   
Smart persistence                      4.599  0.758   14.021    9.754  5.622   
PV history – MLP                       3.672  0.662   10.714    8.924  4.611   
PV history – LSTM                      3.712  0.661   10.723    9.398  4.606   
PV history – Transformer               3.853  0.665   11.431    8.921  4.660   
Clear-sky index history – MLP          3.593  0.672   10.578    8.207  4.356   
Clear-sky index history – LSTM         3.569  0.673   10.405    8.442  4.517   
Clear-sky index history – Transformer  3.600  0.664   10.396    8.908  4.505   
Sky images – CNN                       4.316  1.885    9.367   10.531  4.712   
Concatenation                          3.180  0.727    9.002    7.321  3.875   
Residual (two-step)                    3.242  0.729    9.088    7.770  4.061   
FiLM (early fusion)                    3.258  0.756    9.154    7.352  4.055   
Cross-attention                        3.213  0.724    9.035    7.473  3.913   
Concatenation, pre-trained encoder     3.351  0.746    9.447    7.737  3.933   

horizon                                                             60  \
weather                                clear variable overcast     all   
model                                                                    
Persistence                            6.264   16.352   15.434  14.851   
Smart persistence                      1.187   15.546   14.646   7.072   
PV history – MLP                       0.886   12.718   13.083   5.477   
PV history – LSTM                      0.878   12.630   13.276   5.819   
PV history – Transformer               0.892   12.795   13.178   5.886   
Clear-sky index history – MLP          0.896   12.204   11.270   5.397   
Clear-sky index history – LSTM         0.916   12.471   12.297   5.603   
Clear-sky index history – Transformer  0.895   12.369   12.688   5.500   
Sky images – CNN                       1.959   10.291   12.343   5.634   
Concatenation                          1.003   10.081   10.453   4.899   
Residual (two-step)                    0.990   10.686   11.151   5.099   
FiLM (early fusion)                    1.048   10.449   11.325   4.956   
Cross-attention                        0.991   10.240   10.608   4.759   
Concatenation, pre-trained encoder     1.047   10.224   10.510   4.939   

horizon                                                          
weather                                 clear variable overcast  
model                                                            
Persistence                            12.097   20.168   22.975  
Smart persistence                       1.921   17.348   21.231  
PV history – MLP                        1.190   14.045   17.131  
PV history – LSTM                       1.207   14.649   19.946  
PV history – Transformer                1.223   14.913   19.758  
Clear-sky index history – MLP           1.233   13.803   16.622  
Clear-sky index history – LSTM          1.229   14.222   17.993  
Clear-sky index history – Transformer   1.226   13.650   18.508  
Sky images – CNN                        2.358   11.767   15.852  
Concatenation                           1.351   11.536   16.068  
Residual (two-step)                     1.436   12.226   16.024  
FiLM (early fusion)                     1.377   11.503   16.936  
Cross-attention                         1.312   11.329   15.178  
Concatenation, pre-trained encoder      1.387   11.760   15.573

## 5 · Stage 2: test set

The models selected in stage 1 are trained once more and evaluated on the test set, which is used **only here and only once**:

| Model | Why | On the test set |
|---|---|---|
| Persistence, smart persistence | references (skill score) | no training; σ and ν fitted to the errors of all trainval samples, per weather bin |
| Clear-sky index MLP, clear-sky index LSTM | the two best single-modality models in 4.1 | trained on the whole training set |
| Concatenation, cross-attention | the two best fusion models in 4.2 | trained on the whole training set |

The learned models keep the architecture and training settings of `02_baselines` and `03_fusion` unchanged. Each is trained once on all trainval days except 10% of the days, which are held out for early stopping (chosen as in `02_baselines`: whole days, fixed seed; saved to `evaluation/test/early_stop_days.json`). The inputs are standardised with the training part. The model then predicts the test set once.

The 90% intervals on the test set are calibrated with the weather-conditional q̂ computed from the out-of-fold scores of the **same** model in all 5 folds of the cross-validation. Nothing is changed after the test results have been seen.

### 5.1 Setup and data

In [58]:
import json, math, time
from types import SimpleNamespace

import torch
import torch.nn as nn
import torch.nn.functional as F

TEST_DIR = RESULTS_DIR / "test"
TEST_PRED_DIR = TEST_DIR / "predictions"       # test predictions, one file per model and horizon
TEST_MODEL_DIR = TEST_DIR / "models"           # final models (weights + standardisation)
TEST_LOG_DIR = TEST_DIR / "train_logs"
for p in [TEST_PRED_DIR, TEST_MODEL_DIR, TEST_LOG_DIR]:
    p.mkdir(parents=True, exist_ok=True)
IMAGE_CACHE = PROCESSED_DIR / "images_rgb.npy"

# settings identical to 02_baselines and 03_fusion
HISTORY_MIN = 15
IMAGE_OFFSETS = [10, 5, 0]
SEED = 1
TRAIN_CONFIG = dict(lr=1e-3, weight_decay=1e-4, batch_size=256, max_epochs=60, early_stop_patience=10,
                    lr_patience=3, lr_min=1e-5, grad_clip=1.0)
RETRAIN = False                  # False: skip models whose test predictions already exist

# models on the test set: the two persistence references, the two best single-modality and the two best fusion models
TEST_MODELS = ["persistence", "smart_persistence", "k_mlp", "k_lstm", "fusion_concat", "fusion_xattn"]

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.backends.cudnn.benchmark = True
print("device:", DEVICE, torch.cuda.get_device_name(0) if DEVICE.type == "cuda" else "")

device: cuda NVIDIA GeForce RTX 4070 SUPER


In [59]:
# prepare data from the preprocessing
# sperate data into 3 parts: train, early_stop(validation) and the original offical test
minutes = pd.read_csv(PROCESSED_DIR / "meta_processed.csv", usecols=["date", "pv_kw", "k", "p_clear", "sun_elev", "sun_azim"])
elev, azim = np.deg2rad(minutes["sun_elev"].to_numpy()), np.deg2rad(minutes["sun_azim"].to_numpy())
sun_per_minute = np.stack([np.sin(elev), np.sin(azim), np.cos(azim)], axis=1).astype(np.float32)

# early-stopping days of the final training: 10% of all trainval days, chosen as in 02_baselines (whole days, fixed seed)
split_days = pd.read_csv(PROCESSED_DIR / "split_days.csv")
trainval_days = np.sort(split_days.loc[split_days["fold"] >= 0, "date"].to_numpy())
rng = np.random.default_rng(SEED + N_FOLDS)
FINAL_EARLY_STOP_DAYS = sorted(rng.choice(trainval_days, size=round(0.1 * len(trainval_days)), replace=False))
with open(TEST_DIR / "early_stop_days.json", "w") as fp:
    json.dump(FINAL_EARLY_STOP_DAYS, fp, indent=1)

def load_horizon(h):
    # all samples of one horizon: fit (trainval without the early-stopping days), early stop, test
    s = pd.read_csv(PROCESSED_DIR / f"samples_h{h}.csv")
    t, y = s["row_t"].to_numpy(), s["row_y"].to_numpy()
    pv, k = minutes["pv_kw"].to_numpy(np.float32), minutes["k"].to_numpy(np.float32)
    test = s["fold"].to_numpy() < 0
    early_stop = ~test & np.isin(minutes["date"].to_numpy()[t], FINAL_EARLY_STOP_DAYS)
    return SimpleNamespace(
        samples=s[["row_t", "row_y", "weather"]], weather=s["weather"].to_numpy(),
        fit=~test & ~early_stop, early_stop=early_stop, test=test,
        k_history=k[t[:, None] + np.arange(-HISTORY_MIN + 1, 1)],
        sun=np.concatenate([sun_per_minute[t], sun_per_minute[y]], axis=1),
        image_rows=t[:, None] - np.array(IMAGE_OFFSETS),
        pv_now=pv[t], k_now=k[t], pv_target=pv[y], k_target=k[y],
        clear_sky_target=minutes["p_clear"].to_numpy(np.float32)[y])

DATA = {h: load_horizon(h) for h in HORIZONS}
print(pd.DataFrame({f"{h} min": {part: getattr(DATA[h], part).sum() for part in ["fit", "early_stop", "test"]}
                    for h in HORIZONS}))

            15 min  30 min  60 min
fit         269200  261813  247799
early_stop   31524   30709   29191
test         12118   11818   11218


### 5.2 Models

Copied unchanged from `02_baselines` (Student-t head, MLP and LSTM history encoders, history model) and `03_fusion` (image encoder, routed head, concatenation, cross-attention).

In [60]:
# from 02_baselines: output head, history encoders, history model
# t distribution output head
class StudentTHead(nn.Module):
    def __init__(self, n_in, hidden=128, sigma_min=1e-3):
        super().__init__()
        self.shared = nn.Sequential(nn.Linear(n_in, hidden), nn.ReLU())
        self.mu_head = nn.Sequential(nn.Linear(hidden, hidden // 2), nn.ReLU(), nn.Linear(hidden // 2, 1))
        self.sigma_head = nn.Sequential(nn.Linear(hidden, hidden // 2), nn.ReLU(), nn.Linear(hidden // 2, 1))
        self.nu_raw = nn.Parameter(torch.tensor(1.0))
        self.sigma_min = sigma_min

    def forward(self, features):
        x = self.shared(features)
        mu = self.mu_head(x).squeeze(-1)
        sigma = F.softplus(self.sigma_head(x)).squeeze(-1) + self.sigma_min
        return mu, sigma, 2 + F.softplus(self.nu_raw)

# MLP
class MLPEncoder(nn.Module):
    def __init__(self, seq_len=HISTORY_MIN, hidden=128):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(seq_len, hidden), nn.ReLU(), nn.Linear(hidden, hidden), nn.ReLU())
        self.out_dim = hidden

    def forward(self, history):
        return self.net(history)

# LSTM
class LSTMEncoder(nn.Module):
    def __init__(self, hidden=64, layers=2, dropout=0.1):
        super().__init__()
        self.lstm = nn.LSTM(1, hidden, num_layers=layers, batch_first=True, dropout=dropout)
        self.out_dim = hidden

    def forward(self, history):
        _, (h_n, _) = self.lstm(history.unsqueeze(-1))
        return h_n[-1]

# model for only time-series history data
class HistoryModel(nn.Module):
    use_amp = False

    def __init__(self, encoder):
        super().__init__()
        self.encoder = encoder
        self.head = StudentTHead(encoder.out_dim + 6)

    def forward(self, batch):
        return self.head(torch.cat([self.encoder(batch["history"]), batch["sun"]], dim=1))


#############################################################################################
# from 03_fusion: image encoder, routed head, concatenation, cross-attention
class ResidualBlock(nn.Module):
    def __init__(self, c_in, c_out):
        super().__init__()
        self.conv1 = nn.Conv2d(c_in, c_out, 3, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(c_out)
        self.conv2 = nn.Conv2d(c_out, c_out, 3, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(c_out)
        self.skip = nn.Identity() if c_in == c_out else nn.Sequential(nn.Conv2d(c_in, c_out, 1, bias=False), nn.BatchNorm2d(c_out))

    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)))
        return F.relu(self.bn2(self.conv2(out)) + self.skip(x))

# image encoder for images
class ImageEncoder(nn.Module):
    def __init__(self, in_channels=9, widths=(32, 64, 128), dropout=0.1):
        super().__init__()
        layers = [nn.Conv2d(in_channels, widths[0], 3, padding=1, bias=False), nn.BatchNorm2d(widths[0]), nn.ReLU()]
        c = widths[0]
        for w in widths:
            layers += [ResidualBlock(c, w), nn.MaxPool2d(2), nn.Dropout2d(dropout)]
            c = w
        self.net = nn.Sequential(*layers)
        self.out_channels = c

    def forward(self, images):
        return self.net(images)                          # (batch, 128, 8, 8)

# student t distribution output head with different rounting
class RoutedStudentTHead(nn.Module):
    def __init__(self, n_history, n_image, route="both", hidden=128, sigma_min=1e-3):
        super().__init__()
        self.image_to_mu = route in ("both", "mu")
        self.image_to_sigma = route in ("both", "sigma")

        def branch(n_in):
            return nn.Sequential(nn.Linear(n_in, hidden), nn.ReLU(), nn.Linear(hidden, hidden // 2), nn.ReLU(),
                                 nn.Linear(hidden // 2, 1))
        self.mu_head = branch(n_history + n_image * self.image_to_mu)
        self.sigma_head = branch(n_history + n_image * self.image_to_sigma)
        self.nu_raw = nn.Parameter(torch.tensor(1.0))
        self.sigma_min = sigma_min

    def forward(self, history_features, image_features):
        both = torch.cat([history_features, image_features], dim=1)
        mu = self.mu_head(both if self.image_to_mu else history_features).squeeze(-1)
        sigma = F.softplus(self.sigma_head(both if self.image_to_sigma else history_features)).squeeze(-1) + self.sigma_min
        return mu, sigma, 2 + F.softplus(self.nu_raw)

def image_summary(channels, dropout):
    return nn.Sequential(nn.AdaptiveAvgPool2d(4), nn.Flatten(), nn.Linear(channels * 16, 128), nn.ReLU(), nn.Dropout(dropout))

# concatation
class ConcatFusion(nn.Module):
    use_amp = True                                    # as in 03_fusion: bfloat16 for the fusion models

    def __init__(self, route="both", dropout=0.3):
        super().__init__()
        self.history_encoder = MLPEncoder()
        self.image_encoder = ImageEncoder()
        self.summary = image_summary(self.image_encoder.out_channels, dropout)
        self.head = RoutedStudentTHead(self.history_encoder.out_dim + 6, 128, route)

    def forward(self, batch):
        history_features = torch.cat([self.history_encoder(batch["history"]), batch["sun"]], dim=1).float()
        image_features = self.summary(self.image_encoder(batch["images"])).float()
        return self.head(history_features, image_features)

# cross attention
class CrossAttentionBlock(nn.Module):
    def __init__(self, d_model, heads, dropout):
        super().__init__()
        self.norm_query = nn.LayerNorm(d_model)
        self.attention = nn.MultiheadAttention(d_model, heads, dropout=dropout, batch_first=True)
        self.feed_forward = nn.Sequential(nn.LayerNorm(d_model), nn.Linear(d_model, 2 * d_model), nn.GELU(),
                                          nn.Dropout(dropout), nn.Linear(2 * d_model, d_model))

    def forward(self, query, tokens, keep_attention=False):
        attended, weights = self.attention(self.norm_query(query), tokens, tokens, need_weights=keep_attention)
        query = query + attended
        return query + self.feed_forward(query), weights

class CrossAttentionFusion(nn.Module):
    use_amp = True

    def __init__(self, route="both", n_queries=4, d_model=128, heads=4, dropout=0.3):
        super().__init__()
        self.history_encoder = MLPEncoder()
        self.image_encoder = ImageEncoder()
        n_history = self.history_encoder.out_dim + 6
        self.summary = image_summary(self.image_encoder.out_channels, dropout)
        self.history_query = nn.Linear(n_history, d_model)
        self.learned_queries = nn.Parameter(torch.randn(1, n_queries - 1, d_model) * 0.02)
        self.token_proj = nn.Linear(self.image_encoder.out_channels, d_model)
        self.token_position = nn.Parameter(torch.randn(1, 64, d_model) * 0.02)
        self.norm_tokens = nn.LayerNorm(d_model)
        self.attention = CrossAttentionBlock(d_model, heads, dropout=0.1)
        self.local = nn.Sequential(nn.Flatten(), nn.Linear(n_queries * d_model, 128), nn.ReLU(), nn.Dropout(dropout))
        self.head = RoutedStudentTHead(n_history, 256, route)
        self.keep_attention = False
        self.last_attention = None

    def forward(self, batch):
        history_features = torch.cat([self.history_encoder(batch["history"]), batch["sun"]], dim=1).float()
        feature_map = self.image_encoder(batch["images"])
        tokens = self.norm_tokens(self.token_proj(feature_map.flatten(2).transpose(1, 2)) + self.token_position)
        queries = torch.cat([self.history_query(history_features).unsqueeze(1),
                             self.learned_queries.expand(len(tokens), -1, -1)], dim=1)
        queries, weights = self.attention(queries, tokens, self.keep_attention)
        if self.keep_attention:
            self.last_attention = weights.reshape(len(tokens), -1, 8, 8).float()
        image_features = torch.cat([self.summary(feature_map), self.local(queries)], dim=1).float()
        return self.head(history_features, image_features)

# name -> (build the model, uses images)
FINAL_MODELS = {
    "k_mlp": (lambda: HistoryModel(MLPEncoder()), False),
    "k_lstm": (lambda: HistoryModel(LSTMEncoder()), False),
    "fusion_concat": (ConcatFusion, True),
    "fusion_xattn": (CrossAttentionFusion, True),
}

### 5.3 Training and test prediction

The training loop, the image augmentation and the inputs are the same as in `02_baselines` and `03_fusion`. `train_final` trains one model on the whole training set, predicts the test set (converted back to kW) and saves the test predictions, the model and the training log to `evaluation/test/`.

In [61]:
def gpu(a, dtype=torch.float32):
    return torch.as_tensor(np.ascontiguousarray(a), dtype=dtype, device=DEVICE)

def student_t_nll(mu, sigma, nu, y):
    z = (y - mu) / sigma
    return (torch.lgamma(nu / 2) - torch.lgamma((nu + 1) / 2) + 0.5 * torch.log(nu * math.pi) + torch.log(sigma)
            + (nu + 1) / 2 * torch.log1p(z ** 2 / nu)).mean()

# forward pass
def forward_pass(model, batch):
    with torch.autocast("cuda", dtype=torch.bfloat16, enabled=DEVICE.type == "cuda" and model.use_amp):
        mu, sigma, nu = model(batch)
    return mu.float(), sigma.float(), nu.float()

# predict
@torch.no_grad()
def predict(model, get_batch, idx, chunk=1024):
    model.eval()
    mus, sigmas = [], []
    for i in range(0, len(idx), chunk):
        mu, sigma, nu = forward_pass(model, get_batch(idx[i:i + chunk]))
        mus.append(mu)
        sigmas.append(sigma)
    return torch.cat(mus), torch.cat(sigmas), nu

# trainning
def train_model(model, get_batch, target, fit_idx, stop_idx):
    # identical to 02_baselines and 03_fusion
    cfg = TRAIN_CONFIG
    opt = torch.optim.AdamW(model.parameters(), lr=cfg["lr"], weight_decay=cfg["weight_decay"])
    sched = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, factor=0.5, patience=cfg["lr_patience"], min_lr=cfg["lr_min"])
    best_nll, best_epoch, best_state, bad_epochs = np.inf, 0, None, 0
    for epoch in range(cfg["max_epochs"]):
        model.train()
        order = fit_idx[torch.randperm(len(fit_idx), device=DEVICE)]
        for i in range(0, len(order), cfg["batch_size"]):
            # forward
            idx = order[i:i + cfg["batch_size"]]
            mu, sigma, nu = forward_pass(model, get_batch(idx, train=True))

            # get result and loss
            y = target[idx]
            loss = F.mse_loss(mu, y) if epoch == 0 else student_t_nll(mu, sigma, nu, y)

            # backward
            opt.zero_grad()
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), cfg["grad_clip"])
            opt.step()
        mu, sigma, nu = predict(model, get_batch, stop_idx)
        stop_nll = student_t_nll(mu, sigma, nu, target[stop_idx]).item()
        if epoch == 0:
            continue
        sched.step(stop_nll)
        if stop_nll < best_nll:
            best_nll, best_epoch, bad_epochs = stop_nll, epoch, 0
            best_state = {n: v.detach().clone() for n, v in model.state_dict().items()}
        else:
            bad_epochs += 1
            if bad_epochs >= cfg["early_stop_patience"]:
                break
    model.load_state_dict(best_state)
    return dict(epochs=epoch + 1, best_epoch=best_epoch, early_stop_nll=round(best_nll, 4))

# image augmentation and input images (from 03_fusion)
def augment_frames(x):
    b, dev = x.shape[0], x.device
    brightness = torch.empty(b, 1, 1, 1, 1, device=dev).uniform_(0.8, 1.2)
    contrast = torch.empty(b, 1, 1, 1, 1, device=dev).uniform_(0.8, 1.2)
    mean = x.mean(dim=(1, 2, 3, 4), keepdim=True)
    x = (x - mean) * contrast + mean * brightness + 0.02 * torch.randn_like(x)
    theta = torch.zeros(b, 2, 3, device=dev)
    theta[:, 0, 0] = 1
    theta[:, 1, 1] = 1
    theta[:, :, 2] = torch.empty(b, 2, device=dev).uniform_(-2 / 32, 2 / 32)
    flat = x.flatten(1, 2)
    grid = F.affine_grid(theta, list(flat.shape), align_corners=False)
    flat = F.grid_sample(flat, grid, padding_mode="zeros", align_corners=False)
    return flat.view_as(x).clamp(0, 1)

def make_images(frames, augment=False):
    x = frames.float().div_(255)
    if augment:
        x = augment_frames(x)
    return x.flatten(1, 2)

IMAGES = None                                         # sky images on the GPU, loaded when a fusion model is trained

def prepare(data, with_images):
    # inputs standardised with the fit part (as in 02_baselines and 03_fusion); target k at t+h
    global IMAGES
    fit, history, target = data.fit, data.k_history, data.k_target
    norm = dict(history_mean=history[fit].mean(), history_std=history[fit].std(),
                sun_mean=data.sun[fit].mean(0), sun_std=data.sun[fit].std(0) + 1e-6,
                y_mean=target[fit].mean(), y_std=target[fit].std())
    history_z = gpu((history - norm["history_mean"]) / norm["history_std"])
    sun_z = gpu((data.sun - norm["sun_mean"]) / norm["sun_std"])
    if with_images and IMAGES is None:
        IMAGES = torch.from_numpy(np.load(IMAGE_CACHE)).to(DEVICE)
    image_rows = gpu(data.image_rows, torch.long)

    def get_batch(idx, train=False):
        batch = {"history": history_z[idx], "sun": sun_z[idx]}
        if with_images:
            batch["images"] = make_images(IMAGES[image_rows[idx]], augment=train)   # augment training batches only
        return batch
    return get_batch, gpu((target - norm["y_mean"]) / norm["y_std"]), norm

def test_file(model, h):
    return TEST_PRED_DIR / f"{model}_h{h}.csv"

def save_test(model, h, mu, sigma, nu):
    data = DATA[h]
    pred = data.samples[data.test].copy()
    pred["y"], pred["mu"], pred["sigma"], pred["nu"] = data.pv_target[data.test], mu, sigma, nu
    assert not pred[["mu", "sigma", "nu"]].isna().any().any()
    pred.to_csv(test_file(model, h), index=False)

def train_final(name, h):
    # train once on the whole training set, then predict the test set
    build_model, with_images = FINAL_MODELS[name]
    data, start = DATA[h], time.time()
    get_batch, target, norm = prepare(data, with_images)
    torch.manual_seed(SEED * 100 + N_FOLDS)
    model = build_model().to(DEVICE)
    info = train_model(model, get_batch, target, gpu(np.flatnonzero(data.fit), torch.long),
                       gpu(np.flatnonzero(data.early_stop), torch.long))
    mu, sigma, nu = predict(model, get_batch, gpu(np.flatnonzero(data.test), torch.long))
    to_kw = data.clear_sky_target[data.test]
    save_test(name, h, (mu.cpu().numpy() * norm["y_std"] + norm["y_mean"]) * to_kw,
              sigma.cpu().numpy() * norm["y_std"] * to_kw, nu.item())
    torch.save({"state_dict": model.state_dict(), "norm": norm}, TEST_MODEL_DIR / f"{name}_h{h}.pt")
    info.update(model=name, horizon=h, nu=round(nu.item(), 3), minutes=round((time.time() - start) / 60, 2))
    pd.DataFrame([info]).to_csv(TEST_LOG_DIR / f"{name}_h{h}.csv", index=False)
    print(f"{name}, h={h}: {info['epochs']} epochs (best {info['best_epoch']}), "
          f"early-stop NLL {info['early_stop_nll']:.3f}, nu {info['nu']:.2f}, {info['minutes']:.1f} min")
    del model
    torch.cuda.empty_cache()

### 5.4 Persistence references on the test set

In [62]:
# use persistence and smart persistence to predict test
# see the baseline performance on test set
def fit_error_distribution(errors, rng, max_n=50_000):
    # Student-t centred at 0 fitted to past errors (as in 02_baselines); returns (nu, sigma)
    e = rng.choice(errors, size=min(len(errors), max_n), replace=False)
    nu, _, sigma = stats.t.fit(e, floc=0)
    if nu < 2.05:
        nu = 2.05
        _, _, sigma = stats.t.fit(e, f0=nu, floc=0)
    return nu, sigma

PERSISTENCE = {"persistence": lambda d: d.pv_now, "smart_persistence": lambda d: d.k_now * d.clear_sky_target}
for name, forecast in PERSISTENCE.items():
    rng = np.random.default_rng(SEED)
    for h in HORIZONS:
        if test_file(name, h).exists() and not RETRAIN:
            print(f"{name}, h={h}: already done, skipped")
            continue
        data = DATA[h]
        mu = forecast(data)
        error = data.pv_target - mu
        sigma, nu = np.zeros_like(mu), np.zeros_like(mu)
        for w in WEATHER_BINS:                        # error distribution of all trainval samples of this weather bin
            nu_w, sigma_w = fit_error_distribution(error[~data.test & (data.weather == w)], rng)
            sigma[data.weather == w], nu[data.weather == w] = sigma_w, nu_w
        save_test(name, h, mu[data.test], sigma[data.test], nu[data.test])
        print(f"{name}, h={h}: done")

persistence, h=15: already done, skipped
persistence, h=30: already done, skipped
persistence, h=60: already done, skipped
smart_persistence, h=15: already done, skipped
smart_persistence, h=30: already done, skipped
smart_persistence, h=60: already done, skipped


### 5.5 Train the final models

Four models at three horizons.

In [63]:
for name in FINAL_MODELS:
    for h in HORIZONS:
        if test_file(name, h).exists() and not RETRAIN:
            print(f"{name}, h={h}: already done, skipped")
            continue
        train_final(name, h)

k_mlp, h=15: already done, skipped
k_mlp, h=30: already done, skipped
k_mlp, h=60: already done, skipped
k_lstm, h=15: already done, skipped
k_lstm, h=30: already done, skipped
k_lstm, h=60: already done, skipped
fusion_concat, h=15: already done, skipped
fusion_concat, h=30: already done, skipped
fusion_concat, h=60: already done, skipped
fusion_xattn, h=15: already done, skipped
fusion_xattn, h=30: already done, skipped
fusion_xattn, h=60: already done, skipped


### 5.6 Conformal calibration and metrics on the test set

Same metrics as in stage 1, for the raw interval and the global and weather-conditional conformal intervals; q̂ from the out-of-fold scores of all 5 folds. The results are saved to `evaluation/test_metrics.csv`.

In [64]:
# get the q for the 90% interval
def test_intervals(test, oof, calibration):
    # 90% interval of every test sample: "raw" = Student-t interval of the model;
    # "global" / "weather" = conformal, q from the out-of-fold scores of all 5 folds (02_baselines / 03_fusion)
    if calibration == "raw":
        q = stats.t.ppf(1 - ALPHA / 2, test["nu"].to_numpy())
    else:
        score = ((oof["y"] - oof["mu"]).abs() / oof["sigma"]).to_numpy()
        q = np.full(len(test), np.nan)
        for w in (WEATHER_BINS if calibration == "weather" else [None]):
            calib = np.ones(len(oof), bool) if w is None else (oof["weather"] == w).to_numpy()
            target = np.ones(len(test), bool) if w is None else (test["weather"] == w).to_numpy()
            q[target] = conformal_quantile(score[calib])
    mu, sigma = test["mu"].to_numpy(), test["sigma"].to_numpy()
    return mu - q * sigma, mu + q * sigma

def evaluate_test(test, oof):
    y, mu, sigma, nu = (test[c].to_numpy() for c in ["y", "mu", "sigma", "nu"])
    z = (y - mu) / sigma
    crps, nll = sigma * crps_student_t(z, nu), np.log(sigma) - stats.t.logpdf(z, nu)
    bounds = {c: test_intervals(test, oof, c) for c in ["raw", "global", "weather"]}
    rows = []
    for weather in ["all"] + WEATHER_BINS:
        m = (test["weather"] == weather).to_numpy() | (weather == "all")
        point = {"RMSE": np.sqrt(np.mean((y[m] - mu[m]) ** 2)), "MAE": np.mean(np.abs(y[m] - mu[m])),
                 "CRPS": crps[m].mean(), "NLL": nll[m].mean()}
        for calibration, (low, high) in bounds.items():
            lo, hi, yy = low[m], high[m], y[m]
            miss = (lo - yy) * (yy < lo) + (yy - hi) * (yy > hi)
            rows.append({"weather": weather, "calibration": calibration, "n": int(m.sum()), **point,
                         "coverage": np.mean((yy >= lo) & (yy <= hi)), "width": np.mean(hi - lo),
                         "Winkler": np.mean(hi - lo + 2 / ALPHA * miss)})
    return pd.DataFrame(rows)

test_results = []
for h in HORIZONS:
    for model in TEST_MODELS:
        r = evaluate_test(pd.read_csv(test_file(model, h)), load_predictions(model, h))
        r.insert(0, "model", model)
        r.insert(1, "horizon", h)
        test_results.append(r)
test_results = pd.concat(test_results, ignore_index=True)

key = ["horizon", "weather", "calibration"]
reference = test_results[test_results["model"] == "smart_persistence"].set_index(key)["RMSE"]
test_results["skill"] = 1 - test_results["RMSE"] / reference.reindex(pd.MultiIndex.from_frame(test_results[key])).to_numpy()
test_results.to_csv(RESULTS_DIR / "test_metrics.csv", index=False)
print(f"{test_results['model'].nunique()} models, {len(test_results)} rows")

6 models, 216 rows


### 5.7 Test results

All metrics on the test set after weather-conditional calibration (all weather).

In [65]:
r = test_results[(test_results["calibration"] == "weather") & (test_results["weather"] == "all")]
t = r.set_index(["horizon", "model"])[METRICS].rename(index=LABELS, level="model").round(3)
for h in HORIZONS:
    print(f"\n{h} minutes")
    display(t.loc[h])


15 minutes


,RMSE,MAE,skill,CRPS,NLL,coverage,width,Winkler
model,,,,,,,,
Persistence,3.886,2.213,-0.029,1.699,2.276,0.859,7.764,16.379
Smart persistence,3.778,1.749,0.000,1.430,1.475,0.873,6.361,15.243
Clear-sky index history – MLP,3.363,1.593,0.110,1.181,0.871,0.908,7.177,11.414
Clear-sky index history – LSTM,3.389,1.587,0.103,1.184,0.896,0.910,7.190,11.488
Concatenation,3.094,1.491,0.181,1.068,0.782,0.910,6.954,9.027
Cross-attention,3.137,1.515,0.170,1.088,0.787,0.912,6.900,9.125



30 minutes


,RMSE,MAE,skill,CRPS,NLL,coverage,width,Winkler
model,,,,,,,,
Persistence,4.616,3.084,-0.082,2.264,2.705,0.860,10.826,19.514
Smart persistence,4.266,2.127,0.000,1.697,1.851,0.852,7.753,17.408
Clear-sky index history – MLP,3.710,1.826,0.130,1.369,1.135,0.905,7.900,13.548
Clear-sky index history – LSTM,3.810,1.895,0.107,1.430,1.206,0.904,9.108,14.512
Concatenation,3.651,1.821,0.144,1.312,1.068,0.905,7.951,10.976
Cross-attention,3.581,1.786,0.161,1.290,1.054,0.911,7.348,10.387



60 minutes


,RMSE,MAE,skill,CRPS,NLL,coverage,width,Winkler
model,,,,,,,,
Persistence,5.750,4.465,-0.251,3.128,3.102,0.873,16.182,22.888
Smart persistence,4.597,2.522,0.000,1.935,2.203,0.838,9.601,18.435
Clear-sky index history – MLP,3.967,2.076,0.137,1.542,1.449,0.918,9.575,15.126
Clear-sky index history – LSTM,4.030,2.148,0.123,1.605,1.536,0.915,11.531,16.437
Concatenation,3.917,2.015,0.148,1.497,1.353,0.904,8.151,14.008
Cross-attention,3.757,1.995,0.183,1.438,1.368,0.919,8.112,12.117


### 5.8 Coverage by weather on the test set

Coverage of the raw interval and of the weather-conditional conformal interval in each weather bin. The q̂ come from the cross-validation, so 90% on the test set is not guaranteed: this checks whether the calibration still holds on new days with a new model.

In [66]:
def test_by_weather(metric, calibration):
    r = test_results[test_results["calibration"] == calibration]
    t = r.groupby(["model", "horizon", "weather"], sort=False)[metric].mean().unstack(["horizon", "weather"])
    return t.reindex(columns=["all"] + WEATHER_BINS, level=1).rename(index=LABELS).round(3)

for calibration in ["raw", "weather"]:
    print(f"coverage, {calibration} intervals")
    display(test_by_weather("coverage", calibration))

coverage, raw intervals


horizon                            15                              30         \
weather                           all  clear variable overcast    all  clear   
model                                                                          
Persistence                     0.881  0.965    0.790    0.771  0.888  0.955   
Smart persistence               0.843  0.938    0.730    0.738  0.847  0.923   
Clear-sky index history – MLP   0.906  0.934    0.871    0.882  0.909  0.944   
Clear-sky index history – LSTM  0.903  0.937    0.865    0.856  0.892  0.923   
Concatenation                   0.891  0.897    0.887    0.879  0.881  0.891   
Cross-attention                 0.891  0.906    0.874    0.872  0.896  0.913   

horizon                                              60                  \
weather                        variable overcast    all  clear variable   
model                                                                     
Persistence                       0.810    0.809  0.909  0.949    0.868   
Smart persistence                 0.752    0.774  0.863  0.906    0.808   
Clear-sky index history – MLP     0.863    0.879  0.907  0.942    0.857   
Clear-sky index history – LSTM    0.849    0.880  0.904  0.933    0.857   
Concatenation                     0.859    0.903  0.880  0.926    0.810   
Cross-attention                   0.871    0.889  0.910  0.932    0.877   

horizon                                  
weather                        overcast  
model                                    
Persistence                       0.850  
Smart persistence                 0.830  
Clear-sky index history – MLP     0.894  
Clear-sky index history – LSTM    0.911  
Concatenation                     0.884  
Cross-attention                   0.908

coverage, weather intervals


horizon                            15                              30         \
weather                           all  clear variable overcast    all  clear   
model                                                                          
Persistence                     0.859  0.882    0.832    0.833  0.860  0.888   
Smart persistence               0.873  0.907    0.836    0.828  0.852  0.881   
Clear-sky index history – MLP   0.908  0.925    0.884    0.898  0.905  0.936   
Clear-sky index history – LSTM  0.910  0.932    0.880    0.894  0.904  0.923   
Concatenation                   0.910  0.909    0.914    0.902  0.905  0.909   
Cross-attention                 0.912  0.918    0.911    0.889  0.911  0.923   

horizon                                              60                  \
weather                        variable overcast    all  clear variable   
model                                                                     
Persistence                       0.825    0.833  0.873  0.888    0.853   
Smart persistence                 0.814    0.827  0.838  0.831    0.837   
Clear-sky index history – MLP     0.868    0.872  0.918  0.944    0.886   
Clear-sky index history – LSTM    0.876    0.898  0.915  0.931    0.891   
Concatenation                     0.891    0.925  0.904  0.941    0.840   
Cross-attention                   0.893    0.909  0.919  0.943    0.883   

horizon                                  
weather                        overcast  
model                                    
Persistence                       0.869  
Smart persistence                 0.872  
Clear-sky index history – MLP     0.895  
Clear-sky index history – LSTM    0.917  
Concatenation                     0.929  
Cross-attention                   0.914

### 5.9 By weather on the test set

RMSE, CRPS, NLL and the width of the weather-conditional interval (kW) in each weather bin. In stage 1 only the width is shown by weather (4.3), because RMSE, CRPS and NLL by weather are already in `02_baselines` and `03_fusion`; for the test set they are shown here.

In [67]:
for metric in ["RMSE", "CRPS", "NLL", "width"]:
    print(f"{metric} by horizon and weather")
    display(test_by_weather(metric, "weather"))

RMSE by horizon and weather


horizon                            15                              30         \
weather                           all  clear variable overcast    all  clear   
model                                                                          
Persistence                     3.886  1.554    5.671    4.866  4.616  2.405   
Smart persistence               3.778  1.138    5.622    4.835  4.266  1.245   
Clear-sky index history – MLP   3.363  1.112    4.977    4.284  3.710  1.172   
Clear-sky index history – LSTM  3.389  1.093    5.003    4.389  3.810  1.206   
Concatenation                   3.094  0.872    4.548    4.191  3.651  1.128   
Cross-attention                 3.137  0.922    4.568    4.338  3.581  1.075   

horizon                                              60                  \
weather                        variable overcast    all  clear variable   
model                                                                     
Persistence                       6.353    6.045  5.750  4.196    7.105   
Smart persistence                 6.206    5.906  4.597  1.607    6.472   
Clear-sky index history – MLP     5.373    5.118  3.967  1.444    5.672   
Clear-sky index history – LSTM    5.497    5.312  4.030  1.401    5.744   
Concatenation                     5.187    5.342  3.917  1.303    5.521   
Cross-attention                   5.130    5.155  3.757  1.240    5.282   

horizon                                  
weather                        overcast  
model                                    
Persistence                       7.140  
Smart persistence                 6.567  
Clear-sky index history – MLP     5.336  
Clear-sky index history – LSTM    5.555  
Concatenation                     5.645  
Cross-attention                   5.462

CRPS by horizon and weather


horizon                            15                              30         \
weather                           all  clear variable overcast    all  clear   
model                                                                          
Persistence                     1.699  0.712    3.013    2.401  2.264  1.309   
Smart persistence               1.430  0.235    2.989    2.371  1.697  0.356   
Clear-sky index history – MLP   1.181  0.168    2.506    1.966  1.369  0.220   
Clear-sky index history – LSTM  1.184  0.169    2.497    2.011  1.430  0.237   
Concatenation                   1.068  0.140    2.232    1.923  1.312  0.210   
Cross-attention                 1.088  0.144    2.256    2.005  1.290  0.206   

horizon                                              60                  \
weather                        variable overcast    all  clear variable   
model                                                                     
Persistence                       3.472    3.100  3.128  2.395    3.990   
Smart persistence                 3.353    2.982  1.935  0.558    3.524   
Clear-sky index history – MLP     2.790    2.463  1.542  0.304    3.008   
Clear-sky index history – LSTM    2.892    2.609  1.605  0.314    3.108   
Concatenation                     2.612    2.538  1.497  0.267    2.907   
Cross-attention                   2.595    2.422  1.438  0.275    2.759   

horizon                                  
weather                        overcast  
model                                    
Persistence                       3.873  
Smart persistence                 3.424  
Clear-sky index history – MLP     2.765  
Clear-sky index history – LSTM    2.957  
Concatenation                     2.854  
Cross-attention                   2.754

NLL by horizon and weather


horizon                            15                              30         \
weather                           all  clear variable overcast    all  clear   
model                                                                          
Persistence                     2.276  1.631    3.111    2.804  2.705  2.262   
Smart persistence               1.475  0.143    3.120    2.779  1.851  0.710   
Clear-sky index history – MLP   0.871 -0.608    2.715    2.267  1.135 -0.318   
Clear-sky index history – LSTM  0.896 -0.565    2.704    2.311  1.206 -0.236   
Concatenation                   0.782 -0.646    2.530    2.217  1.068 -0.336   
Cross-attention                 0.787 -0.654    2.542    2.267  1.054 -0.331   

horizon                                              60                  \
weather                        variable overcast    all  clear variable   
model                                                                     
Persistence                       3.268    3.084  3.102  2.857    3.388   
Smart persistence                 3.237    3.013  2.203  1.272    3.284   
Clear-sky index history – MLP     2.888    2.642  1.449  0.088    3.022   
Clear-sky index history – LSTM    2.934    2.737  1.536  0.199    3.070   
Concatenation                     2.735    2.597  1.353 -0.062    2.994   
Cross-attention                   2.716    2.516  1.368  0.060    2.864   

horizon                                  
weather                        overcast  
model                                    
Persistence                       3.355  
Smart persistence                 3.188  
Clear-sky index history – MLP     2.912  
Clear-sky index history – LSTM    3.006  
Concatenation                     2.854  
Cross-attention                   2.826

width by horizon and weather


horizon                            15                               30         \
weather                           all  clear variable overcast     all  clear   
model                                                                           
Persistence                     7.764  3.199   14.206   10.019  10.826  6.291   
Smart persistence               6.361  0.759   14.043    9.738   7.753  1.197   
Clear-sky index history – MLP   7.177  0.578   15.519   13.098   7.900  0.855   
Clear-sky index history – LSTM  7.190  0.635   15.392   13.302   9.108  0.960   
Concatenation                   6.954  0.758   14.916   12.159   7.951  1.016   
Cross-attention                 6.900  0.752   14.698   12.346   7.348  0.979   

horizon                                               60                   \
weather                        variable overcast     all   clear variable   
model                                                                       
Persistence                      16.343   15.409  16.182  12.065   20.130   
Smart persistence                15.628   14.661   9.601   1.906   17.343   
Clear-sky index history – MLP    16.892   13.854   9.575   1.178   19.569   
Clear-sky index history – LSTM   18.798   17.960  11.531   1.271   22.834   
Concatenation                    15.586   17.187   8.151   1.182   14.980   
Cross-attention                  14.606   15.146   8.112   1.408   15.180   

horizon                                  
weather                        overcast  
model                                    
Persistence                      22.995  
Smart persistence                21.265  
Clear-sky index history – MLP    17.744  
Clear-sky index history – LSTM   24.191  
Concatenation                    19.246  
Cross-attention                  17.320